# MCP Server Distillation — Tutorial

**Goal:** Automatically generate high-quality tool-use training data to make a small language model an expert at using your custom MCP server.

## The Problem

You've built a custom MCP server (APIs, tools, data sources) and connected it to a small language model (e.g., Qwen3-8B). But the small model struggles:
- Picks the wrong tool when several look similar
- Extracts parameters incorrectly from natural language
- Fails multi-step tasks that require chaining tools together
- Doesn't understand the relationships between your tools

## The Solution: MCP Server Distillation

Instead of manually writing training examples, we use a **frontier model** (e.g., GPT-5.2) to automatically generate expert-quality training data for your specific MCP server.

The pipeline has three key phases:

### Phase 1: Explore
The frontier model connects to your MCP server and **actively calls every tool** — discovering real data entities, learning how tools relate to each other, identifying which tools are easily confused, and finding edge cases. This produces a "server understanding" document that grounds everything that follows.

### Phase 2: Generate
Using the exploration findings, a teacher LLM synthesizes realistic user questions that require multi-tool reasoning. The frontier model then **solves each question** by actually calling your MCP tools, producing expert-quality tool-use trajectories (the exact sequence of tool calls, arguments, and responses).

### Phase 3: Format
The structured tool traces are converted into the **function-calling conversation format** used for supervised fine-tuning — each example becomes a complete conversation with tool declarations, user question, structured `function_call`/`function` message pairs, and a final answer.

```
  ┌───────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
  │  Expert   │   │ Question │   │ Quality  │   │ Expert   │   │ Format   │
  │Exploration│──▶│Synthesis │──▶│ Filter   │──▶│Trajectory│──▶│ Training │
  │           │   │          │   │          │   │          │   │   Data   │
  └───────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────┘
   Frontier model  Teacher LLM   Teacher LLM    Frontier model  Structured
   calls MCP tools generates Qs  scores quality  solves via MCP  messages
```

## What you need

1. **Your MCP server** running (this tutorial uses a demo e-commerce server)

      `
      uv run python examples/agentic/ecommerce_mcp/server.py
      `

2. **Langflow** with a frontier model agent connected to your MCP server
3. **An API key** for the teacher LLM (used for question generation + quality scoring)

---
## 0. Setup

First, let's import dependencies and configure credentials. You'll need to set your Langflow URL and API keys — either in a `.env` file or as environment variables.

In [ ]:
from pathlib import Path
import json
import os
import sys

# Required to run the flow with async mode
import nest_asyncio
import pandas as pd

nest_asyncio.apply()

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

PROJECT_ROOT = NOTEBOOK_DIR.parent.parent.parent
print(f"Notebook dir:  {NOTEBOOK_DIR}")
print(f"Project root:  {PROJECT_ROOT}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")
TEACHER_MODEL = "openai/gpt-5.2"

# Langflow flow with FRONTIER MODEL (GPT-5.2) + MCP server
# This is NOT the student model — the frontier model generates expert trajectories
LANGFLOW_URL = os.environ.get(
    "LANGFLOW_URL",
    "http://localhost:3000/api/v1/run/30271334-5c84-47e7-88cb-7165ca7c561b",
)
LANGFLOW_API_KEY = os.environ.get("LANGFLOW_API_KEY", None)

DATASET_DIR = NOTEBOOK_DIR / "ecommerce_dataset"
CHECKPOINT_DIR = NOTEBOOK_DIR / "checkpoints"

print(f"Teacher model:  {TEACHER_MODEL}")
print(f"Langflow URL:   {LANGFLOW_URL}")

---
## 1. Understand Your MCP Server

Before generating training data, let's understand what we're working with. This tutorial uses the **ShopInsights Analytics Platform** — a demo e-commerce MCP server with 15 tools. In your own workflow, this would be replaced by your custom MCP server.

The tools are deliberately designed with **ambiguity clusters** — groups of tools that look similar but serve different purposes. This is exactly the kind of challenge that small models struggle with and that our distillation pipeline is designed to solve.

| Cluster | Tools | Why it's hard for small models |
|---|---|---|
| Product Discovery | `search_products`, `browse_catalog`, `get_trending_products`, `get_product_details` | Which search/browse tool is right for a given query? |
| Sales & Revenue | `get_sales_data`, `get_revenue_report`, `get_store_overview` | Per-product units vs. aggregate revenue vs. quick snapshot |
| Customer Analytics | `get_customer_segments`, `get_customer_profile`, `get_abandoned_carts` | Individual lookup vs. aggregate analysis vs. behavioral data |
| Multi-step | `analyze_product_performance`, `compare_products`, `forecast_demand`, `get_inventory_status`, `create_promotion` | Require chaining 2-4 tools in the correct order |

In [ ]:
from data import create_data_store

store = create_data_store()

print(f"Products:        {len(store.products)}")
print(f"Customers:       {len(store.customers)}")
print(f"Orders:          {len(store.orders)}")
print(f"Inventory rows:  {len(store.inventory)}")
print(f"Abandoned carts: {len(store.abandoned_carts)}")
print(f"Promotions:      {len(store.promotions)}")

print("\nCategory hierarchy:")
for cat in sorted({p["category"] for p in store.products}):
    count = sum(1 for p in store.products if p["category"] == cat)
    print(f"  {cat} ({count} products)")

In [ ]:
from server import mcp

tools = await mcp.list_tools()

print(f"{'#':<3} {'Tool Name':<32} {'Params':>6}  Description")
print("\u2500" * 100)
for i, t in enumerate(tools, 1):
    mt = t.to_mcp_tool()
    n_params = len(mt.inputSchema.get("properties", {}))
    desc = (mt.description or "").split("\n")[0][:50]
    print(f"{i:<3} {mt.name:<32} {n_params:>6}  {desc}")

In [ ]:
# Quick smoke test: call a few tools directly
from server import get_store_overview, search_products

print("=== Store Overview ===")
overview = get_store_overview()
print(f"  Revenue: ${overview['total_revenue']:,.2f}")
print(f"  Orders:  {overview['total_orders']}")
print(f"  AOV:     ${overview['average_order_value']:,.2f}")
print("  Top products:")
for p in overview["top_5_products"]:
    print(f"    {p['product_id']}: {p['name']} (${p['revenue']:,.2f})")

print("\n=== Search: 'wireless' ===")
results = search_products(query="wireless", limit=3)
for p in results["products"]:
    print(f"  {p['id']}: {p['name']} \u2014 ${p['price']:.2f}")

---
## 2. Create the Input Dataset

The distillation pipeline needs a simple input: a DataFrame with your MCP server's tool schemas, name, and description. This is all the pipeline needs to start generating training data — it will discover everything else through exploration.

| Column | Type | Description |
|---|---|---|
| `tool_list` | `list[dict]` | Each tool's `name`, `description`, and `inputSchema` (JSON Schema) |
| `mcp_server_name` | `str` | Human-readable server name |
| `mcp_server_description` | `str` | What the server does |

In [ ]:
tool_list = []
for t in tools:
    mt = t.to_mcp_tool()
    tool_list.append(
        {
            "name": mt.name,
            "description": mt.description or "",
            "inputSchema": mt.inputSchema,
        }
    )

server_name = mcp.name or "ShopInsights Analytics Platform"
server_description = (
    "E-commerce analytics platform for an online retailer. "
    "Provides product search, sales analytics, customer insights, "
    "demand forecasting, and promotional management. "
    "Features 15 tools organized across product discovery, sales & revenue, "
    "customer analytics, and multi-step analytical workflows."
)

df = pd.DataFrame(
    {
        "tool_list": [tool_list],
        "mcp_server_name": [server_name],
        "mcp_server_description": [server_description],
    }
)

---
## 3. Load the Distillation Flow

The pipeline is defined as a YAML flow with **23 blocks** across **6 stages**. Each stage is composable — you can tune parameters like how many question variants to generate, what quality thresholds to apply, and what timeouts to use for exploration.

| Stage | What happens | Why it matters |
|---|---|---|
| 1. **Expert Exploration** | Frontier model connects to your MCP server and calls every tool | Discovers real data, tool relationships, and edge cases |
| 2. **Diversity** | Multiplies rows and samples different tool subsets | Ensures training data covers diverse tool combinations |
| 3. **Question Synthesis** | Teacher LLM generates realistic questions using exploration findings | Questions reference real entities, not hypothetical ones |
| 4. **Question Quality Filter** | Scores questions on difficulty, realism, uniqueness | Removes trivial or poorly-formed questions |
| 5. **Expert Trajectories** | Frontier model solves each question via actual MCP tool calls | Produces gold-standard demonstrations of correct tool use |
| 6. **Response Quality Filter** | Scores trajectories on completeness and conciseness | Removes incomplete or verbose responses |

In [ ]:
from sdg_hub import Flow, FlowRegistry

# Auto-discover all available flows
FlowRegistry.discover_flows()

In [ ]:
# Every flow gets a deterministic ID
# Same flow name always generates the same ID
flow_id = "new-night-835"

# Use ID to reference the flow
flow_path = FlowRegistry.get_flow_path(flow_id)
flow = Flow.from_yaml(flow_path)

print(f"Flow: {flow.metadata.name}")
print(f"Version: {flow.metadata.version}")
print(f"Tags: {flow.metadata.tags}")
print(f"Blocks: {len(flow.blocks)}")
print(f"Agent blocks: {flow._detect_agent_blocks()}")
print(f"\nRecommended model: {flow.get_model_recommendations()}")

In [ ]:
flow.print_info()

---
## 4. Configure the Pipeline

The flow needs two separate configurations — one for the teacher LLM (API calls for question synthesis and quality scoring), and one for the frontier model agent (Langflow + MCP server for exploration and trajectory generation).

**Key concept:** The Langflow agent should run the **frontier model** (e.g., GPT-5.2), not the student model you're trying to train. The student model is only used *after* training, not during data generation.

In [ ]:
# Configure the teacher model (LLM API calls for question gen + quality scoring)
flow.set_model_config(
    model=TEACHER_MODEL,
    api_key=OPENAI_API_KEY,
)
print(f"Teacher model configured: {TEACHER_MODEL}")

In [ ]:
# Configure the frontier model agent (Langflow + MCP server)
# Both agent blocks (explore_server + run_expert_trajectory) use the same Langflow flow
agent_kwargs = {
    "agent_framework": "langflow",
    "agent_url": LANGFLOW_URL,
}
if LANGFLOW_API_KEY:
    agent_kwargs["agent_api_key"] = LANGFLOW_API_KEY

flow.set_agent_config(**agent_kwargs)
print(f"Agent configured for: {flow._detect_agent_blocks()}")
print(f"Langflow URL: {LANGFLOW_URL}")

# Optional: longer timeout for exploration (it calls many tools)
flow.set_agent_config(timeout=300, blocks=["explore_server"])
print("Exploration timeout set to 300s")

---
## 5. Run the Pipeline

Now we execute the full pipeline. The `flow.generate()` method runs all 23 blocks sequentially.

**Expected data flow** (starting from 1 row with 15 tools):
```
1 row  →  Exploration (adds server_understanding)
       →  ×10 multiplier = 10 rows
       →  Sample 2-tool subsets per row
       →  Generate questions (10 candidate questions)
       →  Quality filter → ~6-8 questions survive
       →  Expert trajectories (frontier model solves each)
       →  Response quality filter → final training examples
```

The quality filters are intentionally aggressive — we'd rather have fewer, higher-quality examples than many noisy ones. You can adjust the thresholds using runtime overrides.

In [ ]:
print(f"Input: {len(df)} row(s), {len(df['tool_list'][0])} tools")

In [ ]:
# Run the full distillation pipeline
result = flow.generate(df)

In [ ]:
print("Pipeline complete!")
print(f"Generated {len(result)} training examples")
print(
    f"Columns: {result.column_names if hasattr(result, 'column_names') else list(result.columns)}"
)

---
## 6. Exploration Deep Dive

This is the most important step in the pipeline. Unlike approaches that only read tool schemas, our pipeline has the frontier model **actually call the tools** and observe their behavior.

The exploration produces a structured "server understanding" document that captures:
- **Data entities** — real product IDs, customer names, category hierarchies, price ranges
- **Tool relationships** — which tool outputs feed into which tool inputs (e.g., `search_products` returns IDs that `get_product_details` needs)
- **Ambiguity clusters** — which tools overlap in functionality and when to use each one
- **Edge cases** — error conditions, empty results, parameter constraints

This understanding is then used to generate **grounded questions** — questions that reference real entities and exploit known tool relationships, rather than generic hypotheticals.

In [ ]:
if hasattr(result, "to_pandas"):
    df = result.to_pandas()
else:
    df = result

# The server_understanding column contains the exploration findings
if "server_understanding" in df.columns:
    # All rows share the same exploration (it was done once, then multiplied)
    exploration = df["server_understanding"].iloc[0]
    print("=" * 80)
    print("SERVER UNDERSTANDING (from frontier model exploration)")
    print("=" * 80)
    print(exploration[:3000])
    if len(exploration) > 3000:
        print(f"\n... ({len(exploration) - 3000} more characters)")
else:
    print("server_understanding column not found \u2014 check pipeline output above")
    print(f"Available columns: {list(df.columns)}")

---
## 7. Results Analysis

Let's look at what the pipeline produced. Each surviving row contains:
- A **realistic question** grounded in exploration findings
- The **target tools** that should be used to answer it
- An **expert trajectory** — the frontier model's actual tool calls and final answer
- **Quality scores** from the teacher LLM (question quality, completeness, conciseness)

In [ ]:
# Key columns summary
key_cols = [
    c
    for c in [
        "question",
        "target_tools",
        "question_quality_rating",
        "completeness_rating",
    ]
    if c in df.columns
]
if key_cols:
    display(df[key_cols])

In [ ]:
# Key columns summary
key_cols = [
    c
    for c in [
        "question",
        "target_tools",
        "question_quality_rating",
        "completeness_rating",
        "conciseness_rating",
    ]
    if c in df.columns
]
if key_cols:
    display(df[key_cols])

In [ ]:
# Quality distribution
for col in ["question_quality_rating", "completeness_rating"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts().to_string())

---
## 8. Format Training Data

The pipeline now outputs a `extract_agent_text_tool_trace` column — the full structured tool call trace extracted directly from the Langflow agent response. Each trace contains:

- **Input** — the user's question
- **Tool calls** — each tool the expert called, with exact arguments and structured output
- **Output** — the final synthesized answer

We convert these traces into the **function-calling conversation format** used for SFT:

```
[system]    "Here are the tools you can use: [tool schemas...]"
[user]      "Find the top trending laptop and check inventory"
[assistant] → function_call: get_trending_products(metric="revenue", ...)
[function]  ← {"trending": [{"id": "PROD-0006", ...}]}
[assistant] → function_call: get_inventory_status(product_ids=["PROD-0006"])
[function]  ← {"inventory": [...], "low_stock_count": 0}
[assistant] "Here's what I found: the AeroBook Gaming X17 is trending..."
```

In [ ]:
# Step 1: Inspect the tool traces the pipeline already extracted
#
# The flow's AgentResponseExtractorBlock has extract_tool_trace=True,
# so the structured trace is already in the output — no post-processing needed.

trace = df["extract_agent_text_tool_trace"].iloc[0]
print(f"First example has {len(trace)} trace steps:")
for i, step in enumerate(trace):
    stype = step.get("type")
    if stype == "tool_use":
        print(f"  [{i}] tool_use → {step['name']}()")
    else:
        title = step.get("header", {}).get("title", "")
        print(f"  [{i}] text [{title}]")

In [ ]:
# Step 2: Import the formatter
#
# tool_trace_to_messages() converts a Langflow tool trace + tool schemas into
# the structured function-calling conversation format used for SFT training.
#
# Each trace step maps to one or two messages:
#   "text" (Input)  → {"role": "user", ...}
#   "tool_use"      → {"role": "assistant", "function_call": {...}}
#                     {"role": "function", "content": ..., "name": ...}
#   "text" (Output) → {"role": "assistant", ...}

from sdg_hub.core.utils.message_formatter import tool_trace_to_messages

print("Imported: tool_trace_to_messages()")

In [ ]:
# Step 3: Format all rows into structured conversations
#
# For each training example, we combine:
#   - tool_list (schemas)  →  system message with full tool declarations
#   - tool_trace (steps)   →  user question + function_call/function pairs + final answer

formatted_messages = []
for idx, row in df.iterrows():
    trace = row["extract_agent_text_tool_trace"]
    tools = row["tool_list"]
    msgs = tool_trace_to_messages(trace, tools)
    formatted_messages.append(json.dumps(msgs))

df["messages"] = formatted_messages

print(f"Formatted {len(df)} rows into structured conversations")
for idx, row in df.iterrows():
    msgs = json.loads(row["messages"])
    roles = [m["role"] for m in msgs]
    n_tool_calls = sum(1 for m in msgs if "function_call" in m)
    print(
        f"  Row {idx}: {len(msgs)} messages, {n_tool_calls} tool call(s) — roles: {roles}"
    )

In [ ]:
# Step 4: Inspect a formatted example
#
# Each training example is now a complete conversation that teaches the model:
#   1. What tools are available (system message)
#   2. What the user asked (user message)
#   3. Which tools to call and with what arguments (assistant function_call)
#   4. What the tools returned (function messages)
#   5. How to synthesize the results into a final answer (assistant message)

sample_msgs = json.loads(df["messages"].iloc[0])

print(f"Example conversation ({len(sample_msgs)} messages):\n")
for i, msg in enumerate(sample_msgs):
    role = msg["role"]

    if role == "system":
        content = msg["content"]
        tools_start = content.find("[")
        tools_end = content.rfind("]") + 1
        declared = json.loads(content[tools_start:tools_end])
        print(f"[{i}] system — declares {len(declared)} tools")

    elif "function_call" in msg:
        fc = msg["function_call"]
        print(f"\n[{i}] assistant → calls {fc['name']}()")
        print(f"    arguments: {fc['arguments'][:100]}")

    elif role == "function":
        content = msg["content"]
        print(f"[{i}] function ({msg['name']}) → {len(content)} chars")
        print(f"    {content[:100]}...")

    elif role == "user":
        print(f"\n[{i}] user")
        print(f"    {msg['content'][:120]}")

    elif role == "assistant":
        print(f"\n[{i}] assistant (final answer)")
        print(f"    {msg['content'][:200]}...")

---
## 9. Export

The final training data has a `messages` column containing structured function-calling conversations ready for SFT. We export the columns that matter for training.

In [ ]:
# Select the columns that matter for training
export_cols = [
    "messages",
    "question",
    "target_tools",
    "question_quality_rating",
    "completeness_rating",
]
df_export = df[[c for c in export_cols if c in df.columns]]

output_path = NOTEBOOK_DIR / "distillation_results.parquet"
df_export.to_parquet(output_path, index=False)

print(f"Exported {len(df_export)} training examples to {output_path.name}")
print(f"  Columns: {list(df_export.columns)}")
print(f"  Size: {output_path.stat().st_size / 1024:.1f} KB")

# Also save as JSONL for easy inspection
jsonl_path = NOTEBOOK_DIR / "distillation_results.jsonl"
df_export.to_json(jsonl_path, orient="records", lines=True)
print(f"  JSONL: {jsonl_path.name}")

---
## 10. Scaling Up: Generating More Data

The pipeline is designed to scale. You can increase the volume and complexity of generated training data without modifying the flow YAML—everything is configurable through **runtime parameter overrides** passed to `flow.generate()`.

#### Key parameters to tune

| Block                | Parameter       | Default | What it controls                                                  |
|----------------------|----------------|---------|-------------------------------------------------------------------|
| `multiply_tool_rows` | `num_samples`  | 10      | How many question candidates to generate per MCP server row        |
| `sample_tools`       | `num_samples`  | 2       | How many tools each question must involve (higher = harder questions) |

#### Example: 50 examples with 3-tool questions

```python
result = flow.generate(
    df,
    runtime_params={
        "multiply_tool_rows": {"num_samples": 50},  # 50 candidates instead of 10
        "sample_tools": {"num_samples": 3},         # 3-tool combos instead of 2
    },
    checkpoint_dir="./checkpoints",
)
```

### What to expect

- **More candidates → more survivors:**  
  With `num_samples=50`, you'll generate 50 candidate questions. After quality filtering, expect ~25–35 to survive, producing significantly more training data from a single pipeline run.
- **Higher tool count → harder questions:**  
  Setting `sample_tools.num_samples=3` or `4` forces the teacher LLM to generate questions that require chaining more tools together. This produces training data for the hardest scenarios—exactly where small models struggle most.
- **Diminishing returns above 4–5 tools:**  
  Most realistic user queries require 2–4 tools. Sampling 5+ tools per question can lead to contrived scenarios that get filtered out by the quality check.

### Multiple MCP servers

If you have multiple MCP servers, create a multi-row input DataFrame—one row per server. The pipeline will explore and generate data for each server independently:

```python
df = pd.DataFrame({
    "tool_list": [server1_tools, server2_tools, server3_tools],
    "mcp_server_name": ["Payments API", "Inventory API", "Customer API"],
    "mcp_server_description": ["...", "...", "..."],
})
# The exploration phase runs once per server, then questions are generated for each
result = flow.generate(df, runtime_params={"multiply_tool_rows": {"num_samples": 50}})
```

---

## What we built

This tutorial walked through the full **MCP Server Distillation** pipeline — an automated way to generate high-quality tool-use training data for any MCP server.

### The pipeline

1. **Expert Exploration** — A frontier model connected to your MCP server and actively called every tool, discovering real data entities, tool relationships, ambiguity clusters, and edge cases
2. **Informed Question Synthesis** — A teacher LLM generated realistic questions grounded in the exploration findings, not just schema descriptions
3. **Expert Trajectories** — The frontier model solved each question by actually calling your MCP tools, producing gold-standard demonstrations of correct tool use
4. **Quality Filtering** — Questions and responses were scored and filtered to ensure only high-quality examples survive
5. **Training Data Formatting** — Raw tool traces were converted into structured function-calling conversations (system + user + function_call/function pairs + assistant) ready for SFT

### What makes this different

- **Active exploration, not passive schema reading** — The model discovers real data and tool behavior by calling tools, not just reading descriptions
- **Expert trajectories, not student filtering** — A strong model generates consistently good examples, rather than filtering for rare good outputs from a weak model
- **Grounded questions** — Questions reference real entities and exploit known tool relationships, producing more realistic training scenarios
- **Works with any MCP server** — Just point the pipeline at your Langflow agent and MCP server

### What's next?

Now that you've seen how the pipeline works, you can use it to generate training data for your own MCP servers. Then using the generated data, you can train a small model to use the tools effectively.